# Deliverable C — interventional counterfactual PDs

For each query, estimate `p_cf = Pr(y=1 | do(f=v), X_{-f}=x_{-f})`: set one feature to the intervention value, **hold all other features fixed**, and read PD from the calibrated model.

**Why this is the interventional (not naive) estimate.** Holding all other observed features fixed is a back-door adjustment: under selection-on-observables (no unobserved confounding given X_{-f}), `Pr(y|do(f=v),X_{-f}) = Pr(y|f=v,X_{-f})`. The naive failure the brief warns about is perturbing a feature *without* conditioning on its confounders; here the full covariate set is conditioned on by construction. Residual threats (unobserved confounding; off-manifold interventions such as a bank-feed value for a no-feed applicant) are reflected as wider intervals, not hidden.

Output: `submission_C_counterfactuals.csv` (one row per query).

In [ ]:
# ---- configuration ----
# Point DATA_DIR at the folder with train.csv / validation.csv / test.csv,
# and OUT_DIR at where submission files should be written.
DATA_DIR = "."
OUT_DIR  = "."
SEED     = 7
N_ENSEMBLE = 12          # bootstrap members          # bootstrap members (epistemic intervals)

In [ ]:
import os, numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
rng = np.random.default_rng(SEED)

# NOTE ON THE MODEL CHOICE
# We use sklearn's HistGradientBoostingClassifier as the spine: it natively
# handles the informative-missing (MNAR) bank-feed NaNs and the integer-coded
# categoricals, and runs with no extra dependencies. It is a drop-in for a
# tuned LightGBM (see the companion Optuna notebook); swap the estimator in
# `new_pd_model` if you prefer LightGBM + Optuna.

In [ ]:
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val   = pd.read_csv(os.path.join(DATA_DIR, "validation.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
print("shapes:", train.shape, val.shape, test.shape)

In [ ]:
iq = pd.read_csv(os.path.join(DATA_DIR, "intervention_queries.csv"))
print(iq.shape); iq.head()

## 1. Data prep (shared) — integer-coded categoricals for easy intervention

In [ ]:
OUTCOME = ["default_flag","days_to_default","days_to_full_repayment","repayment_status",
           "final_recovered_amount","observation_status"]
DROP = ["business_id","applicant_id","application_timestamp","prior_decision"] + OUTCOME
CAT  = ["sector","geography_region","employee_count_bucket","intended_use_of_funds",
        "owner_personal_credit_band","application_channel"]
def make_X(df):
    X = df.drop(columns=[c for c in DROP if c in df.columns]).copy()
    X["has_linked_bank_feed"] = X["has_linked_bank_feed"].astype(float)
    for c in CAT: X[c] = X[c].astype("int64")     # int codes -> trivial to overwrite on intervention
    return X
FEATS = make_X(train).columns.tolist(); CAT_MASK = [c in CAT for c in FEATS]
obs = train[train.default_flag.notna()].copy(); Xo, yo = make_X(obs), obs.default_flag.astype(int).values
vobs = val[val.default_flag.notna()].copy();   Xv, yv = make_X(vobs), vobs.default_flag.astype(int).values

## 2. Build factual and counterfactual feature frames for the queries

All query applicants live in the test set. For each query we copy the applicant's real feature vector and overwrite the single intervened feature ( `do(f=v)` ).

In [ ]:
base = test.set_index("applicant_id").loc[iq.applicant_id].reset_index()   # 1 row per query
X_factual = make_X(base)
X_cf = X_factual.copy()
for f in iq.feature_name.unique():
    mask = (iq.feature_name == f).values
    vals = iq.loc[mask, "intervention_value"].values
    X_cf.loc[mask, f] = vals.astype("int64") if f in CAT else vals
print("queries:", len(iq), "| below historical cutoff:", int((base.prior_underwriter_score < 0.273).sum()))

## 3. Calibrated ensemble → counterfactual PD with 90% intervals

In [ ]:
def new_pd_model(seed):
    return HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=60,
        l2_regularization=1.0, categorical_features=CAT_MASK, random_state=seed)
raw_cf = np.zeros((N_ENSEMBLE, len(X_cf))); raw_fa = np.zeros((N_ENSEMBLE, len(X_factual)))
raw_v  = np.zeros((N_ENSEMBLE, len(Xv)))
for b in range(N_ENSEMBLE):
    idx = rng.integers(0, len(Xo), len(Xo)); m = new_pd_model(100+b).fit(Xo.iloc[idx], yo[idx])
    raw_cf[b] = m.predict_proba(X_cf)[:,1]; raw_fa[b] = m.predict_proba(X_factual)[:,1]
    raw_v[b]  = m.predict_proba(Xv)[:,1]; print(f"  member {b+1}/{N_ENSEMBLE}")
iso = IsotonicRegression(out_of_bounds="clip").fit(raw_v.mean(0), yv); cal = lambda x: np.clip(iso.predict(x),1e-4,1-1e-4)
cf_members = np.vstack([cal(raw_cf[b]) for b in range(N_ENSEMBLE)])
cf = cal(raw_cf.mean(0)); fa = cal(raw_fa.mean(0))
lo = np.minimum(np.percentile(cf_members,5,0), cf); hi = np.maximum(np.percentile(cf_members,95,0), cf)

## 4. Validate and write `submission_C_counterfactuals.csv`

In [ ]:
C = pd.DataFrame({"query_id": iq.query_id.values, "predicted_pd_cf": np.round(cf,6),
                  "pd_cf_lower_90": np.round(lo,6), "pd_cf_upper_90": np.round(hi,6)})
assert len(C) == len(iq) and C.query_id.is_unique
assert (C.pd_cf_lower_90 <= C.predicted_pd_cf + 1e-9).all()
assert (C.predicted_pd_cf <= C.pd_cf_upper_90 + 1e-9).all() and C.predicted_pd_cf.between(0,1).all()
C.to_csv(os.path.join(OUT_DIR, "submission_C_counterfactuals.csv"), index=False)
print("wrote submission_C_counterfactuals.csv  rows =", len(C))
# directional sanity: mean intervention effect by feature (avg over the query values)
eff = pd.Series(cf - fa).groupby(iq.feature_name.values).mean().sort_values()
print("\nmost PD-lowering / PD-raising interventions:"); print(eff.head(3).round(4)); print(eff.tail(3).round(4))
C.head()